# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 01: Data Collection & Initial Quality Control (QC)

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Target: Tokopedia, Shopee, Lazada, Bukalapak (15,000 Raw Reviews)
```

---
### 1. Deskripsi Metodologis
Tahap ini bertujuan untuk mengumpulkan korpus ulasan pengguna Google Play Store dari 4 platform e-commerce terkemuka di Indonesia:
1. **Tokopedia** (`com.tokopedia.tkpd`)
2. **Shopee** (`com.shopee.id`)
3. **Lazada** (`com.lazada.android`)
4. **Bukalapak** (`com.bukalapak.android`)

Target pengumpulan adalah **15.000 ulasan mentah** (~3.750 ulasan per aplikasi) dengan distribusi rating berimbang (skor 1 sampai 5 masing-masing 750 ulasan per aplikasi) untuk mengeliminasi bias distribusi mayoritas bintang 5.

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11
print("Environment initialized successfully.")

### 2. Eksekusi Scraper Multi-App Berimbang
Scraper diimplementasikan pada modul `scraper/gplay_scraper.py` dengan penanganan *exponential backoff* dan pagination rating.

In [ ]:
# Pengecekan file mentah pada direktori data/raw
raw_files = glob.glob("../data/raw/*_reviews.csv") or glob.glob("data/raw/*_reviews.csv")
print(f"File CSV mentah ditemukan: {len(raw_files)}")
for rf in raw_files:
    print(f"  - {rf}")

### 3. Load & Aggregasi Data Mentah

In [ ]:
data_dir = "../data/raw" if os.path.exists("../data/raw") else "data/raw"
apps = ["tokopedia", "shopee", "lazada", "bukalapak"]
df_list = []

for app in apps:
    filepath = os.path.join(data_dir, f"{app}_reviews.csv")
    if os.path.exists(filepath):
        temp_df = pd.read_csv(filepath)
        df_list.append(temp_df)
        print(f"Loaded {app}: {len(temp_df)} baris")
    else:
        print(f"Menunggu file: {filepath}")

if df_list:
    raw_df = pd.concat(df_list, ignore_index=True)
    print(f"\nTotal Korpus Mentah Tergabung: {len(raw_df)} baris")
    print(raw_df.head(3))
else:
    print("Belum ada data mentah yang dapat dimuat.")

### 4. Quality Control (QC) & Distribusi Skor

In [ ]:
if "raw_df" in locals() and not raw_df.empty:
    # Ringkasan Missing Values
    print("--- Missing Values Check ---")
    print(raw_df.isnull().sum())
    
    # Distribusi Skor per Aplikasi
    plt.figure(figsize=(10, 5))
    sns.countplot(data=raw_df, x="score", hue="app_name", palette="viridis")
    plt.title("Distribusi Rating per Aplikasi (Raw Google Play Store Corpus)")
    plt.xlabel("Skor Rating Bintang (1 - 5)")
    plt.ylabel("Jumlah Ulasan")
    plt.legend(title="Aplikasi")
    plt.tight_layout()
    plt.show()
    
    # Karakteristik Panjang Teks
    raw_df["char_length"] = raw_df["content"].fillna("").astype(str).apply(len)
    print("\n--- Statistik Panjang Karakter Ulasan ---")
    print(raw_df["char_length"].describe())